# Bronze Layer



## Load Population Data

In [0]:
population_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/appalachia/source_system/census_population_county_2020_2025.csv")
    )


## Inspect Population Data

In [0]:
population_df.printSchema()
population_df.select(
    "STNAME",
    "CTYNAME",
    "POPESTIMATE2025"
).sample(0.01).show(10)

## Create Bronze Population Table

In [0]:
population_df.write.mode("overwrite").saveAsTable("workspace.appalachia.bronze_population")

## Testing Population Table

In [0]:
spark.sql("""
SELECT
    STNAME,
    CTYNAME,
    POPESTIMATE2025
FROM workspace.appalachia.bronze_population
LIMIT 10
""").show()

## Load Broadband Data
Broadband year values are derived from source filenames because the raw CSV files do not contain a year column.

In [0]:
from pyspark.sql.functions import regexp_extract, col

broadband_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/appalachia/source_system/broadband_tract_map_dec_*.csv")
    .withColumn("source_file", col("_metadata.file_path"))    
    .withColumn(
       "year",
        regexp_extract(col("source_file"), r"(\d{4})", 1).cast("int")
        )
    )

## Inspect Broadband Data

In [0]:
broadband_df.printSchema()
broadband_df.sample(0.01).show(10)

## Create Bronze Broadband Table

In [0]:
(
    broadband_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.appalachia.bronze_broadband")
)

## Testing Broadband Table

In [0]:
spark.sql("""
          SELECT *
          FROM workspace.appalachia.bronze_broadband
          LIMIT 10
          """).show()

%md
## Load QCEW Data

In [0]:
qcew_paths = [
    "/Volumes/workspace/appalachia/source_system/2021.annual.singlefile.csv.gz",
    "/Volumes/workspace/appalachia/source_system/2022.annual.singlefile.csv.gz",
    "/Volumes/workspace/appalachia/source_system/2024.annual.singlefile.csv.gz",
    "/Volumes/workspace/appalachia/source_system/qcew_2023_part_*.csv"
]

qcew_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(qcew_paths)
)

## Inspect QCEW Data

In [0]:
qcew_df.printSchema()
qcew_df.select(
    "area_fips",
    "industry_code",
    "year",
    "qtr",
    "annual_avg_emplvl",
    "total_annual_wages"
).sample(0.001).show(10)

## Create Bronze QCEW Table

In [0]:
qcew_df.write.mode("overwrite").saveAsTable("workspace.appalachia.bronze_qcew")

## Testing QCEW Table

In [0]:
%sql
SELECT
    area_fips,
    industry_code,
    year,
    qtr, 
    annual_avg_emplvl,
    total_annual_wages
FROM workspace.appalachia.bronze_qcew
LIMIT 10;


## Query for Testing

In [0]:
%sql
SELECT DISTINCT
  industry_code
FROM
  workspace.appalachia.bronze_qcew
WHERE
    agglvl_code = 70 OR
    agglvl_code = 74
ORDER BY
    industry_code;